In [13]:
import pandas as pd
import numpy as np
import warnings
import os
warnings.filterwarnings("ignore")

# Loading data

In [14]:
features_30 = pd.read_csv("./Data/features_30_sec.csv")
features_3 = pd.read_csv("./Data/features_3_sec.csv")

In [243]:
features_3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9990 entries, 0 to 9989
Data columns (total 60 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   filename                 9990 non-null   object 
 1   length                   9990 non-null   int64  
 2   chroma_stft_mean         9990 non-null   float64
 3   chroma_stft_var          9990 non-null   float64
 4   rms_mean                 9990 non-null   float64
 5   rms_var                  9990 non-null   float64
 6   spectral_centroid_mean   9990 non-null   float64
 7   spectral_centroid_var    9990 non-null   float64
 8   spectral_bandwidth_mean  9990 non-null   float64
 9   spectral_bandwidth_var   9990 non-null   float64
 10  rolloff_mean             9990 non-null   float64
 11  rolloff_var              9990 non-null   float64
 12  zero_crossing_rate_mean  9990 non-null   float64
 13  zero_crossing_rate_var   9990 non-null   float64
 14  harmony_mean            

In [244]:
features_30.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 60 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   filename                 1000 non-null   object 
 1   length                   1000 non-null   int64  
 2   chroma_stft_mean         1000 non-null   float64
 3   chroma_stft_var          1000 non-null   float64
 4   rms_mean                 1000 non-null   float64
 5   rms_var                  1000 non-null   float64
 6   spectral_centroid_mean   1000 non-null   float64
 7   spectral_centroid_var    1000 non-null   float64
 8   spectral_bandwidth_mean  1000 non-null   float64
 9   spectral_bandwidth_var   1000 non-null   float64
 10  rolloff_mean             1000 non-null   float64
 11  rolloff_var              1000 non-null   float64
 12  zero_crossing_rate_mean  1000 non-null   float64
 13  zero_crossing_rate_var   1000 non-null   float64
 14  harmony_mean             

### Feature extraction , train test split

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [245]:
scaler = StandardScaler()

In [246]:
features_30[features_30.columns[2:59]] = scaler.fit_transform(features_30.drop(columns=["length","label","filename"]))

In [247]:
features_30.replace({"blues" : 0 , "classical" : 1 , "country" : 2 , "disco" : 3 , "hiphop" : 4 ,
                     "jazz" : 5 , "metal" : 6 , "pop" : 7 , "reggae" : 8 , "rock" : 9},inplace = True)

In [248]:
X_train_30 , X_test_30 , Y_train_30 , Y_test_30  = train_test_split(features_30.drop(columns=["length","label","filename"]),features_30["label"],random_state = 42,test_size=.1)

In [249]:
scaler = StandardScaler()

In [250]:
features_3[features_3.columns[2:59]] = scaler.fit_transform(features_3.drop(columns=["length","label","filename"]))

In [251]:
features_3["file"] = features_3["filename"].apply(lambda x:x.replace(x[-6:],""))

In [263]:
X_train_3 , X_test_3 = train_test_split(features_3["file"].unique(),test_size=.1,random_state=42)

In [264]:
cond1 = [val in X_train_3 for val in features_3["file"]]
cond2 = [val in X_test_3 for val in features_3["file"]]

In [265]:
features_3.replace({"blues" : 0 , "classical" : 1 , "country" : 2 , "disco" : 3 , "hiphop" : 4 ,
                     "jazz" : 5 , "metal" : 6 , "pop" : 7 , "reggae" : 8 , "rock" : 9},inplace = True)

In [266]:
X_train_3 = features_3[cond1].drop(columns=["length","label","filename","file"])
X_test_3 = features_3[cond2].drop(columns=["length","label","filename","file"])
Y_train_3 = features_3[cond1]["label"]
Y_test_3 = features_3[cond2]["label"]

# Tabluar approch (XGBoost model)

In [16]:
from xgboost import XGBClassifier

### Training with 30 seconds features 

In [257]:
xgb_30 = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.1)

In [258]:
xgb_30.fit(X_train_30,Y_train_30)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=None, num_parallel_tree=None, ...)

In [259]:
xgb_30.score(X_train_30,Y_train_30)

0.9988888888888889

In [260]:
xgb_30.score(X_test_30,Y_test_30)

0.78

### Training with 3 seconds features 

In [267]:
xgb_3 = XGBClassifier(n_estimators=500, max_depth=9, learning_rate=0.1)

In [268]:
xgb_3.fit(X_train_3,Y_train_3)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=9,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=None, num_parallel_tree=None, ...)

In [269]:
xgb.score(X_train_3,Y_train_3)

0.9988877766655544

In [270]:
xgb.score(X_test_3,Y_test_3)

0.7687687687687688

# Image-based approch (CNN model)

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.optim as optim

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [21]:
def crop_white_borders(img):
    width, height = img.size
    return TF.crop(img, top=35, left=53, height=height-70, width=width-106)

In [22]:
transform = transforms.Compose([
    transforms.Lambda(crop_white_borders),
    transforms.ToTensor(),
])

In [24]:
train_dataset = datasets.ImageFolder("./Data/images_original/train", transform=transform)
test_dataset  = datasets.ImageFolder("./Data/images_original/test", transform=transform)

In [38]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

In [40]:
print("Classes:", train_dataset.classes)
print("Class mapping:", train_dataset.class_to_idx)

Classes: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
Class mapping: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}


In [41]:
class Net(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)

        # محاسبه‌ی خروجی لایه‌های کانولوشنی به صورت دینامیک
        self._to_linear = None
        self._get_conv_output()

        self.fc1 = nn.Linear(self._to_linear, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def _get_conv_output(self):
        x = torch.zeros(1, 3, 128, 128)
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        self._to_linear = x.numel()

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, self._to_linear)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

In [42]:
model = Net(10).to(device)

In [43]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [46]:
for epoch in range(20):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f} | Test Accuracy: {acc:.2f}%")

RuntimeError: DataLoader worker (pid(s) 24176, 18856) exited unexpectedly